# Lab 3.14 — Adaptation Review
Compare scratch, frozen transfer, and conservative fine-tuning under one controlled target split and scorecard.


In [ ]:
import copy, random, numpy as np, torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
SEED=13; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)
d=load_digits(); X=d.data.astype('float32')/16.0; y=d.target.astype('int64'); Xt,Xv,yt,yv=train_test_split(X,y,test_size=0.25,random_state=SEED,stratify=y)
class E(torch.nn.Module):
    def __init__(self): super().__init__(); self.net=torch.nn.Sequential(torch.nn.Linear(64,32),torch.nn.ReLU())
    def forward(self,x): return self.net(x)
class M(torch.nn.Module):
    def __init__(self,e=None,out=2): super().__init__(); self.encoder=e or E(); self.head=torch.nn.Linear(32,out)
    def forward(self,x): return self.head(self.encoder(x))
def train(m,a,b,n,lr):
    o=torch.optim.SGD([p for p in m.parameters() if p.requires_grad],lr=lr); f=torch.nn.CrossEntropyLoss()
    for _ in range(n): o.zero_grad(); z=f(m(a),b); z.backward(); o.step()
    return m
def acc(m,a,b):
    m.eval();
    with torch.no_grad(): return float((m(a).argmax(1)==b).float().mean())
src=train(M(out=10),torch.tensor(Xt),torch.tensor(yt),120,0.1); target=np.isin(yt,[0,6,8,9]).astype('int64'); tv=torch.tensor(np.isin(yv,[0,6,8,9]).astype('int64')); idx=np.arange(len(yt)); small,_=train_test_split(idx,train_size=40,random_state=SEED,stratify=target); a=torch.tensor(Xt[small]); b=torch.tensor(target[small]); val=torch.tensor(Xv)
torch.manual_seed(SEED); scratch=train(M(),a,b,200,0.2); e=copy.deepcopy(src.encoder); [setattr(p,'requires_grad',False) for p in e.parameters()]; torch.manual_seed(SEED); frozen=train(M(e),a,b,200,0.2)
base=torch.cat([p.detach().flatten() for p in frozen.encoder.parameters()]); fine=copy.deepcopy(frozen); [setattr(p,'requires_grad',True) for p in fine.encoder.parameters()]; fine=train(fine,a,b,80,0.01); move=float(torch.linalg.vector_norm(torch.cat([p.detach().flatten() for p in fine.encoder.parameters()])-base))
rows=[]
for name,m in [('scratch',scratch),('frozen',frozen),('fine_tuned',fine)]:
    rows.append((name,acc(m,val,tv),sum(p.numel() for p in m.parameters() if p.requires_grad),move if name=='fine_tuned' else 0.0))
print('strategy | accuracy | trainable_parameters | encoder_movement')
for r in rows: print(f'{r[0]:10s} | {r[1]:.3f} | {r[2]:5d} | {r[3]:.3f}')
assert all(r[1]>0.75 for r in rows) and rows[1][2]<rows[0][2]


## Decision
Choose one strategy using at least three pieces of evidence: validation quality, trainable parameters, encoder movement, failure slices, and reproducibility cost.
